# VINO omar/ -- Full Training (B1 x {NH, MR, AB}, B2 x {NH, MR, AB})
**Colab / T4 GPU**

Runs the exact same training loop as `VINO_Hyperelasticity.py` (Adam,
1000 epochs, unchanged `utils/fno_utils.train_fno`) for all 6 cases. The
only geometry-specific code lives in `Practical_Examples/omar/` -- the FNO
architecture, optimizer, and training loop are untouched upstream code.

After: **Runtime > Change runtime type > T4 GPU**, then run cells in order.


## Cell 1 - Install JAX (CUDA 12) + deps

In [ ]:
!pip install -q "jax[cuda12]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
!pip install -q flax optax tqdm matplotlib scipy geomdl
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
print('Done - Restart Runtime, then start from Cell 2')


## Cell 2 - Clone the repo

Replace `<TOKEN>` with a GitHub personal access token if the repo is
private (Settings > Developer settings > Personal access tokens). If the
repo is public, drop the `<TOKEN>@` part.

Always does a clean re-clone (removes any previous `/content/OMAR` first)
so a broken partial clone from an earlier failed run can't linger.


In [ ]:
import os
import shutil
import sys

# cd out of /content/OMAR *before* possibly deleting it -- deleting the
# process's current directory out from under it causes
# "getcwd: cannot access parent directories" on any subsequent relative
# path operation (os.getcwd(), os.chdir('.'), etc).
os.chdir('/content')

REPO_URL = "https://<TOKEN>@github.com/suhibamro/omar.git"  # <-- fill in your token (or drop it if public)
BRANCH = "claude/claude-code-question-d307wp"

if os.path.exists('/content/OMAR'):
    shutil.rmtree('/content/OMAR')

!git clone -b {BRANCH} {REPO_URL} /content/OMAR

WORK_DIR = '/content/OMAR/Practical_Examples'
os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
os.makedirs('./data', exist_ok=True)
os.makedirs('./results', exist_ok=True)

utils_ok = os.path.isdir(os.path.join(WORK_DIR, 'utils'))
omar_ok = os.path.isdir(os.path.join(WORK_DIR, 'omar'))
if utils_ok and omar_ok:
    print('Clone OK: utils/ and omar/ both found under Practical_Examples/.')
else:
    print('ERROR: clone looks incomplete.')
    print(f'  utils/ found: {utils_ok}')
    print(f'  omar/  found: {omar_ok}')
    print('Check REPO_URL / BRANCH / token above, then re-run this cell.')

import jax
jax.config.update('jax_enable_x64', True)
print(f'JAX {jax.__version__} | backend={jax.default_backend()} | devices={jax.devices()}')


## Cell 3 - Verify setup

In [ ]:
import os

WORK_DIR = '/content/OMAR/Practical_Examples'
DATA_DIR = os.path.join(WORK_DIR, 'data')

EXPECTED_NPZ = [
    'B1_n550_neohookean_64X64.npz', 'B1_n550_mooneyrivlin_64X64.npz', 'B1_n550_arrudaboyce_64X64.npz',
    'B2_n550_neohookean_64X64.npz', 'B2_n550_mooneyrivlin_64X64.npz', 'B2_n550_arrudaboyce_64X64.npz',
]

dir_checks = {
    'utils/': os.path.isdir(os.path.join(WORK_DIR, 'utils')),
    'omar/': os.path.isdir(os.path.join(WORK_DIR, 'omar')),
    'data/': os.path.isdir(DATA_DIR),
}
npz_checks = {f: os.path.isfile(os.path.join(DATA_DIR, f)) for f in EXPECTED_NPZ}

print('=== Setup check ===')
for name, ok in dir_checks.items():
    print(f'  {name:8s} {"OK" if ok else "MISSING"}')
print('  data/ contents:')
for f, ok in npz_checks.items():
    print(f'    {f:40s} {"OK" if ok else "MISSING"}')

code_ok = dir_checks['utils/'] and dir_checks['omar/']
data_ok = dir_checks['data/'] and all(npz_checks.values())

if code_ok and data_ok:
    print('\nAll checks passed. FEM data already present -- you can skip Cell 4 and go to Cell 5 (training).')
elif code_ok and not data_ok:
    print('\nRepo structure OK, but FEM data is missing/incomplete.')
    print('Run Cell 4 to generate it, or upload the missing .npz files into Practical_Examples/data/.')
    raise SystemExit('Setup check: FEM data missing -- see messages above.')
else:
    print('\nSTOP: repo structure is incomplete. Go back and fix Cell 2 (clone) before continuing.')
    raise SystemExit('Setup check failed -- see messages above.')


## Cell 4 - Generate FEM ground-truth data (FEniCSx)

Only needs to run once; each of the 6 `.npz` files is skipped if it
already exists (see `omar/generate_fem_data.py`). Takes ~15-20 min total
for all 6 cases x 550 samples on Colab's CPU (FEniCSx doesn't use the
GPU; the *training* below does).

If you already have the 6 `.npz` files from a previous run, upload them
to `Practical_Examples/data/` instead and skip straight to Cell 5.


In [ ]:
try:
    import dolfinx
    print('FEniCSx already installed')
except ImportError:
    !wget -q 'https://fem-on-colab.github.io/packages/fenicsx/release/fenicsx-install-real.sh' -O '/tmp/fenicsx.sh'
    !bash /tmp/fenicsx.sh
    print('Restart runtime once more (Runtime > Restart Runtime), then re-run this cell to confirm, then continue')


In [ ]:
# Run after the FEniCSx install + restart above.
os.chdir(WORK_DIR)
!python omar/generate_fem_data.py


## Cell 5 - Train all 6 cases

Reuses `omar/train_B1.py` and `omar/train_B2.py` directly -- same
`utils.fno_utils.train_fno` loop, same `jax.example_libraries.optimizers.adam`,
same `num_epoch = 1000` / `batch_size = 50` / `learning_rate = 0.001` from
`omar/config.py` as the original `VINO_Hyperelasticity.py`. Nothing here
duplicates or reimplements that loop.


In [ ]:
import time
import omar.train_B1 as train_B1
import omar.train_B2 as train_B2
from omar.config import B1_NH, B1_MR, B1_AB, B2_NH, B2_MR, B2_AB

CASES = [
    ("B1_NH", train_B1, B1_NH, 0), ("B1_MR", train_B1, B1_MR, 1), ("B1_AB", train_B1, B1_AB, 2),
    ("B2_NH", train_B2, B2_NH, 3), ("B2_MR", train_B2, B2_MR, 4), ("B2_AB", train_B2, B2_AB, 5),
]

results = {}
for name, module, cfg, seed in CASES:
    print(f'\n===== Training {name} (1000 epochs, Adam) =====')
    t0 = time.time()
    model, params, train_losses, test_losses = module.run(cfg, seed=seed)
    print(f'{name} done in {time.time()-t0:.0f}s | final train={train_losses[-1]:.4f} test={test_losses[-1]:.4f}')
    results[name] = {'model': model, 'params': params, 'train_losses': train_losses, 'test_losses': test_losses}


## Cell 6 - Loss curves

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (name, _, _, _) in zip(axes.flat, CASES):
    r = results[name]
    ax.semilogy(np.abs(r['train_losses']), label='Train')
    ax.semilogy(np.abs(r['test_losses']), label='Test')
    ax.set_title(name)
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('./results/loss_curves.png', dpi=150)
plt.show()


## Cell 7 - Save and download results

In [ ]:
import pickle, shutil
from google.colab import files

for name, _, _, _ in CASES:
    with open(f'./results/params_{name}.pkl', 'wb') as f:
        pickle.dump(results[name]['params'], f)

np.savez('./results/losses.npz', **{
    f'{name}_{split}': results[name][f'{split}_losses']
    for name, _, _, _ in CASES for split in ['train', 'test']
})

shutil.make_archive('omar_training_results', 'zip', './results')
files.download('omar_training_results.zip')
print('Saved and downloaded omar_training_results.zip')
